# Phase 5 Stage 1 — Walk-forward backtest results

Loads all 72 audit logs + summaries from `data/backtests/`, builds the master comparison table, and surfaces top combinations across multiple ranking criteria.

**Reading guide:**
- All runs use the same default slippage (base=2¢, taker=4¢, NegRisk=2¢) — Stage 1 holds slippage constant.
- 3 cohorts × 4 buckets × 2 sizing × 3 windows = 72 runs.
- Sharpe is monthly Sharpe × √12. Negative Sharpe = unprofitable at default slippage.
- Deployment ratio: fraction of test window during which any copy position was open.

## Setup

In [ ]:
import json
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BACKTESTS_DIR = ROOT / 'data' / 'backtests'
summaries = []
for fp in sorted(BACKTESTS_DIR.glob('*_summary.json')):
    summaries.append(json.loads(fp.read_text()))
print(f'loaded {len(summaries)} summaries')

def flatten(s):
    return {
        'cohort': s['cohort'],
        'bucket': s['resolution_bucket'],
        'sizing': s['sizing_rule'],
        'window': s['test_window_start'][:4],
        'n_signals': s['n_signals'],
        'total_pnl': s['total_pnl_usd'],
        'total_volume': s['total_volume_usd'],
        'sharpe': s['sharpe_monthly'],
        'sortino': s['sortino_monthly'],
        'max_dd': s['max_drawdown_usd'],
        'win_rate': s.get('win_rate') or 0,
        'profit_factor': s.get('profit_factor') or 0,
        'deploy_ratio': s['deployment_ratio'],
        'sig_per_wk': s['signals_per_week'],
        'n_leaders': s.get('n_distinct_leaders', 0),
        'n_markets': s.get('n_distinct_markets', 0),
        'negrisk_share': s.get('negrisk_signal_share', 0),
    }
df = pd.DataFrame([flatten(s) for s in summaries])
df

## Master comparison table

In [ ]:
master = df.copy()
master['joint'] = master['sharpe'] * master['deploy_ratio']
print('=== master table sorted by Sharpe ===')
cols = ['cohort','bucket','sizing','window','n_signals','total_pnl','sharpe','win_rate','profit_factor','deploy_ratio','sig_per_wk']
master.sort_values('sharpe', ascending=False)[cols].head(20).round(3)

## Sharpe heatmap (cohort × bucket × sizing × window)

In [ ]:
fig, axes = plt.subplots(len(df['window'].unique()), 2, figsize=(14, 12), squeeze=False)
for i, window in enumerate(sorted(df['window'].unique())):
    for j, sizing in enumerate(sorted(df['sizing'].unique())):
        sub = df[(df['window']==window) & (df['sizing']==sizing)]
        pivot = sub.pivot(index='cohort', columns='bucket', values='sharpe')
        pivot = pivot[['2d','7d','30d','60d']]
        ax = axes[i, j]
        im = ax.imshow(pivot.values, cmap='RdYlGn', vmin=-3, vmax=3, aspect='auto')
        ax.set_xticks(range(len(pivot.columns))); ax.set_xticklabels(pivot.columns)
        ax.set_yticks(range(len(pivot.index))); ax.set_yticklabels(pivot.index)
        for r in range(pivot.shape[0]):
            for c in range(pivot.shape[1]):
                v = pivot.iat[r, c]
                if pd.notna(v):
                    ax.text(c, r, f'{v:.1f}', ha='center', va='center',
                            color='black' if abs(v) < 1.5 else 'white', fontsize=10)
        ax.set_title(f'{window} — {sizing}')
fig.suptitle('Sharpe across cohort × bucket × sizing × window', y=1.0)
fig.colorbar(im, ax=axes.ravel().tolist(), label='Sharpe (monthly × √12)')
plt.tight_layout()

## Top 10 by Sharpe

In [ ]:
df.sort_values('sharpe', ascending=False).head(10)[
    ['cohort','bucket','sizing','window','sharpe','total_pnl','win_rate','profit_factor','deploy_ratio','n_signals']
].round(3)

## Top 10 by total PnL

In [ ]:
df.sort_values('total_pnl', ascending=False).head(10)[
    ['cohort','bucket','sizing','window','total_pnl','sharpe','win_rate','deploy_ratio','n_signals']
].round(3)

## Joint metric: Sharpe × deployment_ratio

A high Sharpe with low capital deployment isn't deployable. This joint metric rewards combinations that *both* compound well and use capital.

In [ ]:
df_joint = df.copy()
df_joint['joint'] = df_joint['sharpe'] * df_joint['deploy_ratio']
df_joint.sort_values('joint', ascending=False).head(10)[
    ['cohort','bucket','sizing','window','sharpe','deploy_ratio','joint','total_pnl','n_signals']
].round(3)

## Capital utilisation analysis

Per design 4: 'If you are not using your capital, you are not running the strategy you think you are running.' Plot deployment ratio vs Sharpe to surface combinations that are both deployed AND profitable.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 7))
for cohort in df['cohort'].unique():
    sub = df[df['cohort']==cohort]
    ax.scatter(sub['deploy_ratio'], sub['sharpe'], s=50 + 3 * sub['n_signals'].clip(0, 200),
               alpha=0.6, label=cohort)
ax.axhline(0, color='black', lw=0.5)
ax.axhline(1.0, color='green', lw=0.5, ls='--', label='Sharpe=1 (success bar)')
ax.axvline(0.4, color='blue', lw=0.5, ls=':', label='40% deploy floor')
ax.set_xlabel('deployment_ratio')
ax.set_ylabel('Sharpe (monthly × √12)')
ax.set_title('Deployment vs Sharpe — bubble size ∝ signal count')
ax.legend()
plt.tight_layout()

## Cross-window robustness

A cohort × bucket × sizing combination is robust if it succeeds across multiple test windows (per design 3.8).

In [ ]:
robustness = df.groupby(['cohort','bucket','sizing']).agg(
    sharpe_min=('sharpe','min'),
    sharpe_max=('sharpe','max'),
    sharpe_mean=('sharpe','mean'),
    pnl_total=('total_pnl','sum'),
    n_positive_windows=('sharpe', lambda x: (x>0).sum()),
    n_above_1=('sharpe', lambda x: (x>1.0).sum()),
    avg_deploy=('deploy_ratio','mean'),
).reset_index()
robustness.sort_values(['n_above_1','sharpe_mean'], ascending=[False, False]).round(3)

## Stage 2 candidate selection

Per design 7.2: 'Top 5 from Stage 1 → 60-slippage-combo grid each'. We need 5 combinations to advance. Selection criterion: top by Sharpe × deploy_ratio, with at least 100 OOS signals and positive PnL on ≥ 2 windows.

In [ ]:
candidates = robustness[(robustness['n_positive_windows']>=2)
                         & (robustness['avg_deploy']>0.05)]
candidates['joint'] = candidates['sharpe_mean'] * candidates['avg_deploy']
candidates.sort_values('joint', ascending=False).head(5)[
    ['cohort','bucket','sizing','sharpe_mean','sharpe_min','sharpe_max',
     'avg_deploy','pnl_total','n_above_1','joint']
].round(3)